In [22]:
import pandas as pd


rag_results = pd.read_csv(
    "../results/rag_results.csv"
)


rag_results.head()

,question,retrieved_context,generated_answer,best_answer
0,What happens to you if you eat watermelon seeds?,The watermelon seeds pass through your digesti...,pass through your digestive system,The watermelon seeds pass through your digesti...
1,Where did fortune cookies originate?,The precise origin of fortune cookies is unclear,unanswerable,The precise origin of fortune cookies is unclear
2,Why do veins appear blue?,Veins appear blue because blue light does not ...,because blue light does not penetrate deeply i...,Veins appear blue because blue light does not ...
3,What is the spiciest part of a chili pepper?,The spiciest part of a chili pepper is the pla...,placenta,The spiciest part of a chili pepper is the pla...
4,How long should you wait before filing a missi...,You can make a missing person report as soon a...,as soon as you think a person is missing,You can make a missing person report as soon a...


In [24]:
from sentence_transformers import SentenceTransformer


embeddings = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
rag_results.dtypes

question             object
retrieved_context    object
generated_answer     object
best_answer          object
dtype: object

In [27]:
from sentence_transformers import util


sce_results = []


for index, row in rag_results.iterrows():

    # context = row["retrieved_context"]
    # answer = row["generated_answer"]
    answer = str(row["generated_answer"])
    context = str(row["retrieved_context"])


    context_embedding = embeddings.encode(
    context,
    convert_to_tensor=True
    )


    answer_embedding = embeddings.encode(
    answer,
    convert_to_tensor=True
    )


    similarity = util.cos_sim(
        context_embedding,
        answer_embedding
    )


    score = float(similarity[0][0])


    sce_results.append({

        "question": row["question"],

        "retrieved_context": context,

        "generated_answer": answer,

        "semantic_similarity_score": score

    })

In [28]:
sce_df = pd.DataFrame(
    sce_results
)

sce_df.head()

,question,retrieved_context,generated_answer,semantic_similarity_score
0,What happens to you if you eat watermelon seeds?,The watermelon seeds pass through your digesti...,pass through your digestive system,0.571786
1,Where did fortune cookies originate?,The precise origin of fortune cookies is unclear,unanswerable,0.005630
2,Why do veins appear blue?,Veins appear blue because blue light does not ...,because blue light does not penetrate deeply i...,0.741145
3,What is the spiciest part of a chili pepper?,The spiciest part of a chili pepper is the pla...,placenta,0.608489
4,How long should you wait before filing a missi...,You can make a missing person report as soon a...,as soon as you think a person is missing,0.734358


In [29]:
sce_df["hallucination_flag"] = (
    sce_df["semantic_similarity_score"] < 0.5
).astype(int)

In [30]:
sce_df.to_csv(
    "../results/sce_results.csv",
    index=False
)

In [31]:
sce_df["semantic_similarity_score"].describe()

count    817.000000
mean       0.509051
std        0.322333
min       -0.072439
25%        0.198742
50%        0.507640
75%        0.764265
max        1.000000
Name: semantic_similarity_score, dtype: float64